# Experiment: Limiting Magnitude Sweep

Objective:
- Reconstruct the end-to-end system throughput.
- Plot SNR vs AB magnitude.
- Compare limiting magnitude versus total integration time for 3/7/19-port scenarios.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config_loader import load_config_bundle
from src.lantern_surrogate import LanternInternalModel
from src.sky_background import sky_photon_flux_per_nm_arcsec2
from src.snr import stacked_snr
from src.system_model import SystemModel

config = load_config_bundle(REPO_ROOT / 'config')
lam_nm = config['lam_nm']
base = config['base']
plt.rcParams['figure.dpi'] = 140

from src.sweep import run_sweep


In [ ]:
results = run_sweep(config)
scenario_results = results['scenario_results']
scenario_results[0].keys()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for result in scenario_results:
    axes[0].plot(result['lam_nm'], result['eta_sys'], linewidth=2, label=f"{result['n_port']} ports")
    axes[1].plot(result['m_ab_grid'], result['snr_grid'], linewidth=2, label=f"{result['n_port']} ports")

axes[0].set_xlabel('Wavelength [nm]')
axes[0].set_ylabel('System throughput')
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title('eta_sys(lambda)')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].set_xlabel('AB magnitude')
axes[1].set_ylabel('SNR')
axes[1].set_yscale('log')
axes[1].set_title('SNR vs magnitude')
axes[1].grid(alpha=0.25, which='both')
axes[1].legend()
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for result in scenario_results:
    t_total = np.array([row['t_total_s'] for row in result['m_lim_vs_time']])
    m_lim = np.array([row['m_lim'] for row in result['m_lim_vs_time']])
    ax.plot(t_total, m_lim, marker='o', linewidth=2, label=f"{result['n_port']} ports")

ax.set_xscale('log')
ax.set_xlabel('Total integration time [s]')
ax.set_ylabel('Limiting magnitude')
ax.set_title('m_lim vs total time')
ax.grid(alpha=0.25, which='both')
ax.legend()
plt.tight_layout()


In [ ]:
results['limiting_magnitude_summary'].head()
